In [1]:
import os
import numpy as np
import pandas as pd
import torch
from itertools import product as iproduct


In [2]:
# ── Cache parameters — must match hm_foaf_experiment_sampled.ipynb ───────────
TARGET_USERS           = 1000
MIN_TRAIN_INTERACTIONS = 10
SEED                   = 42
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    "Cached files not found. Run hm_foaf_experiment_sampled.ipynb first."
)
print(f"Cache tag: {cache_tag}")


Cache tag: u1000_m10_s42


In [3]:
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"]=="n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"]=="n_items", "value"].iloc[0])
print(f"Loaded: {cache_tag}")
print(f"Users: {n_users}  Items: {n_items}")

# min/max from observed train values only — no val/test leakage
r_col      = train_df.columns[2]
min_rating = float(train_df[r_col].min())
max_rating = float(train_df[r_col].max())
print(f"Rating range in train: [{min_rating}, {max_rating}]")
train_df.head()


Loaded: u1000_m10_s42
Users: 474  Items: 5323
Rating range in train: [1.0, 18.0]


,customer_id,product_code,bought
0,194,401,1
1,218,835,1
2,79,498,1
3,12,2042,1
4,344,234,2


In [4]:
# ── Build dense rating matrices (vectorised) ─────────────────────────────────
# Uses pre-split cache CSVs directly — no row-masking split trick.
# Reindexed to [0..n_users) × [0..n_items) for contiguous HM 0-based IDs.
# Unobserved entries → sentinel -1; normalise [0,1] using train stats only.
def df_to_matrix(df, n_u, n_i, lo, hi, sentinel=-1.0):
    u_col, i_col, r_col = df.columns[0], df.columns[1], df.columns[2]
    pivot  = df.pivot_table(index=u_col, columns=i_col,
                            values=r_col, aggfunc="mean")
    pivot  = pivot.reindex(index=range(n_u), columns=range(n_i),
                           fill_value=np.nan)
    mat_np = np.full((n_u, n_i), sentinel, dtype=np.float32)
    obs    = ~pivot.isna().values
    mat_np[obs] = ((pivot.values[obs] - lo) / (hi - lo + 1e-8)).astype(np.float32)
    return torch.tensor(mat_np)

print("Building matrices...")
train_mat = df_to_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_mat   = df_to_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_mat  = df_to_matrix(test_df,  n_users, n_items, min_rating, max_rating)
print(f"train obs : {(train_mat != -1).sum().item()}")
print(f"val   obs : {(val_mat   != -1).sum().item()}")
print(f"test  obs : {(test_mat  != -1).sum().item()}")


Building matrices...
train obs : 15483
val   obs : 3871
test  obs : 7214


In [5]:
# ── SVD model, loss, and RMSE helper ─────────────────────────────────────────
class SVD(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_features, v_features, u_bias, v_bias):
        n_u = u_features.shape[0]
        n_v = v_features.shape[0]
        # Use tensor shapes (not globals); correct broadcast via unsqueeze
        return torch.sigmoid(
            torch.mm(u_features, v_features.t())
            + u_bias.unsqueeze(1).expand(n_u, n_v)
            + v_bias.unsqueeze(0).expand(n_u, n_v)
        )


class SVDLoss(torch.nn.Module):
    def __init__(self, lam_u=0.1, lam_v=0.1, lam_1=0.1, lam_2=0.1):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
        self.lam_1 = lam_1
        self.lam_2 = lam_2

    def forward(self, matrix, predicted, u_features, v_features, u_bias, v_bias):
        mask         = (matrix != -1).float()
        pred_err     = torch.sum((matrix - predicted)**2 * mask)
        u_reg        = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_reg        = self.lam_v * torch.sum(v_features.norm(dim=1))
        ub_reg       = self.lam_1 * u_bias.norm()
        vb_reg       = self.lam_2 * v_bias.norm()
        return pred_err + u_reg + v_reg + ub_reg + vb_reg


def eval_rmse(model, mat, uf, vf, ub, vb, lo, hi):
    """RMSE in original (denormalised) scale over observed entries."""
    with torch.no_grad():
        pred   = model(uf, vf, ub, vb)
        mask   = (mat != -1).float()
        n      = mask.sum()
        pred_d = (pred * (hi - lo) + lo) * mask
        true_d = (mat  * (hi - lo) + lo) * mask
        return float(torch.sqrt(((pred_d - true_d)**2).sum() / n))


def init_params(n_u, n_v, latent, scale=0.01):
    """Initialise all four parameter tensors with small random values."""
    uf = torch.randn(n_u, latent, requires_grad=True)
    vf = torch.randn(n_v, latent, requires_grad=True)
    ub = torch.randn(n_u,         requires_grad=True)
    vb = torch.randn(n_v,         requires_grad=True)
    for t in [uf, vf, ub, vb]:
        t.data.mul_(scale)
    return uf, vf, ub, vb


In [6]:
# ── Step 1 — Parameter tuning (grid search on val set) ───────────────────────
# Searches over latent dimension × learning rate.
# Regularisation coefficients are kept at the paper values; add them to the
# grid if you want to tune them too.
#
# Each config is trained for up to GS_EPOCHS steps with early stopping
# (patience = GS_PATIENCE) to keep the search affordable.

param_grid = {
    "latent": [10, 20, 30],
    "lr":     [0.001, 0.005, 0.01],
    "lam_u":  [0.1],
    "lam_v":  [0.1],
    "lam_1":  [0.1],
    "lam_2":  [0.1],
}

GS_EPOCHS   = 50
GS_PATIENCE = 5

gs_results = []        # (latent, lr, lam_u, lam_v, lam_1, lam_2, best_val_rmse)
best_val   = float("inf")
best_cfg   = None      # (latent, lr, lam_u, lam_v, lam_1, lam_2)

SVD_model = SVD()

for lv, lr, lu, lv0, l1, l2 in iproduct(
        param_grid["latent"], param_grid["lr"],
        param_grid["lam_u"],  param_grid["lam_v"],
        param_grid["lam_1"],  param_grid["lam_2"]):

    uf, vf, ub, vb = init_params(n_users, n_items, lv)
    loss_fn = SVDLoss(lu, lv0, l1, l2)
    opt     = torch.optim.Adam([uf, vf, ub, vb], lr=lr)

    best_sv = float("inf")
    no_imp  = 0
    for step in range(GS_EPOCHS):
        opt.zero_grad()
        pred = SVD_model(uf, vf, ub, vb)
        loss = loss_fn(train_mat, pred, uf, vf, ub, vb)
        loss.backward()
        opt.step()
        vrm = eval_rmse(SVD_model, val_mat, uf, vf, ub, vb, min_rating, max_rating)
        if vrm < best_sv:
            best_sv = vrm
            no_imp  = 0
        else:
            no_imp += 1
        if no_imp >= GS_PATIENCE:
            break

    gs_results.append((lv, lr, lu, lv0, l1, l2, best_sv))
    print(f"latent={lv:2d}  lr={lr:.4f}  val_RMSE={best_sv:.4f}")

    if best_sv < best_val:
        best_val = best_sv
        best_cfg = (lv, lr, lu, lv0, l1, l2)

# Summary table
gs_df = pd.DataFrame(
    gs_results,
    columns=["latent", "lr", "lam_u", "lam_v", "lam_1", "lam_2", "val_rmse"]
).sort_values("val_rmse").reset_index(drop=True)

print(f"\n── Best configuration ──")
print(f"  latent = {best_cfg[0]}")
print(f"  lr     = {best_cfg[1]}")
print(f"  lam_u  = {best_cfg[2]},  lam_v = {best_cfg[3]}")
print(f"  lam_1  = {best_cfg[4]},  lam_2 = {best_cfg[5]}")
print(f"  val RMSE = {best_val:.4f}")
print()
gs_df.head(5)


latent=10  lr=0.0010  val_RMSE=7.7667
latent=10  lr=0.0050  val_RMSE=6.3285
latent=10  lr=0.0100  val_RMSE=4.8772
latent=20  lr=0.0010  val_RMSE=7.7703
latent=20  lr=0.0050  val_RMSE=6.3235
latent=20  lr=0.0100  val_RMSE=3.5924
latent=30  lr=0.0010  val_RMSE=7.7673
latent=30  lr=0.0050  val_RMSE=5.2784
latent=30  lr=0.0100  val_RMSE=2.8400

── Best configuration ──
  latent = 30
  lr     = 0.01
  lam_u  = 0.1,  lam_v = 0.1
  lam_1  = 0.1,  lam_2 = 0.1
  val RMSE = 2.8400



,latent,lr,lam_u,lam_v,lam_1,lam_2,val_rmse
0,30,0.010,0.1,0.1,0.1,0.1,2.840019
1,20,0.010,0.1,0.1,0.1,0.1,3.592442
2,10,0.010,0.1,0.1,0.1,0.1,4.877244
3,30,0.005,0.1,0.1,0.1,0.1,5.278378
4,20,0.005,0.1,0.1,0.1,0.1,6.323495


In [7]:
# ── Step 2 — Full training with best configuration ───────────────────────────
# Trains for up to FULL_EPOCHS steps with early stopping (PATIENCE).
# Best-checkpoint tensors are saved whenever val RMSE improves.

lv, lr, lu, lv0, l1, l2 = best_cfg
FULL_EPOCHS = 100
PATIENCE    = 10

print(f"Training: latent={lv}, lr={lr}, lam_u={lu}, lam_v={lv0}, "
      f"lam_1={l1}, lam_2={l2}")
print(f"Max epochs={FULL_EPOCHS}, early-stopping patience={PATIENCE}")
print()

uf, vf, ub, vb = init_params(n_users, n_items, lv)
SVD_model      = SVD()
loss_fn        = SVDLoss(lu, lv0, l1, l2)
optimizer      = torch.optim.Adam([uf, vf, ub, vb], lr=lr)

best_val_rmse                    = float("inf")
no_imp                           = 0
best_uf = best_vf = best_ub = best_vb = None
val_log                          = []

for step in range(FULL_EPOCHS):
    optimizer.zero_grad()
    pred = SVD_model(uf, vf, ub, vb)
    loss = loss_fn(train_mat, pred, uf, vf, ub, vb)
    loss.backward()
    optimizer.step()

    val_rmse = eval_rmse(SVD_model, val_mat, uf, vf, ub, vb, min_rating, max_rating)
    val_log.append(val_rmse)

    if step % 10 == 0:
        print(f"Step {step:3d} | Train loss: {loss.item():.3f} | Val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_uf = uf.detach().clone()
        best_vf = vf.detach().clone()
        best_ub = ub.detach().clone()
        best_vb = vb.detach().clone()
        no_imp  = 0
    else:
        no_imp += 1

    if no_imp >= PATIENCE:
        print(f"Early stopping at step {step} (best val RMSE = {best_val_rmse:.4f})")
        break


Training: latent=30, lr=0.01, lam_u=0.1, lam_v=0.1, lam_1=0.1, lam_2=0.1
Max epochs=100, early-stopping patience=10

Step   0 | Train loss: 3594.939 | Val RMSE: 8.0805
Step  10 | Train loss: 2871.928 | Val RMSE: 7.2864
Step  20 | Train loss: 1953.595 | Val RMSE: 5.9556
Step  30 | Train loss: 857.400 | Val RMSE: 3.8810
Step  40 | Train loss: 618.684 | Val RMSE: 2.9798
Step  50 | Train loss: 597.811 | Val RMSE: 2.8358
Step  60 | Train loss: 549.506 | Val RMSE: 2.9115
Early stopping at step 60 (best val RMSE = 2.8358)


In [8]:
# ── Step 3 — Test evaluation ─────────────────────────────────────────────────
# Evaluates on the held-out test set using the best-checkpoint parameters.

with torch.no_grad():
    pred_test = SVD_model(best_uf, best_vf, best_ub, best_vb)

non_zero_mask = (test_mat != -1).float()
num           = non_zero_mask.sum()

pred_d = (pred_test * (max_rating - min_rating) + min_rating) * non_zero_mask
true_d = (test_mat  * (max_rating - min_rating) + min_rating) * non_zero_mask

AE_diff   = torch.abs(pred_d - true_d)
SE_diff   = (pred_d - true_d) ** 2

test_MAE  = (AE_diff.sum() / num).item()
test_RMSE = torch.sqrt(SE_diff.sum() / num).item()

print(f"── Test results ──────────────────")
print(f"  Best config : latent={lv}, lr={lr}")
print(f"  Best val RMSE : {best_val_rmse:.4f}")
print(f"  Test MAE    : {test_MAE:.4f}")
print(f"  Test RMSE   : {test_RMSE:.4f}")


── Test results ──────────────────
  Best config : latent=30, lr=0.01
  Best val RMSE : 2.8358
  Test MAE    : 1.2366
  Test RMSE   : 1.8665


In [9]:
# ── Save logs ─────────────────────────────────────────────────────────────────
# Grid search results
gs_df.to_csv("svd_hm_gs.csv", index=False)

# Per-step val RMSE from final training run
log_df = pd.DataFrame({
    "step":     list(range(len(val_log))),
    "val_rmse": val_log,
})
log_df.to_csv("svd_hm.csv", index=False)

print(f"Saved svd_hm_gs.csv  ({len(gs_df)} configs)")
print(f"Saved svd_hm.csv     ({len(log_df)} steps)")
print()
print("Grid search summary (top 5):")
gs_df.head(5)


Saved svd_hm_gs.csv  (9 configs)
Saved svd_hm.csv     (61 steps)

Grid search summary (top 5):


,latent,lr,lam_u,lam_v,lam_1,lam_2,val_rmse
0,30,0.010,0.1,0.1,0.1,0.1,2.840019
1,20,0.010,0.1,0.1,0.1,0.1,3.592442
2,10,0.010,0.1,0.1,0.1,0.1,4.877244
3,30,0.005,0.1,0.1,0.1,0.1,5.278378
4,20,0.005,0.1,0.1,0.1,0.1,6.323495
